# VERA v3: run all, come back later

Run all cells top to bottom. Every heavy step is resumable, so if the session drops just run all again.

1. Keys: put `KAGGLE_USERNAME`, `KAGGLE_KEY`, `HF_TOKEN` in your platform's secrets panel (molab Secrets, Colab Secrets, Kaggle Add-ons > Secrets).
2. Needs a GPU. ~3000 frontal IU CXR studies: extraction takes roughly 2-3 h on a T4.
3. At the end, send Claude the file `vera_work/VERA_v3_results_for_claude.zip`.

In [ ]:
import os
def _load_secret(k):
    if os.environ.get(k):
        return
    try:
        from google.colab import userdata
        os.environ[k] = userdata.get(k)
        return
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ[k] = UserSecretsClient().get_secret(k)
    except Exception:
        pass
for _k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
    _load_secret(_k)
    print(_k, "set" if os.environ.get(_k) else "MISSING")

In [ ]:
import sys
from pathlib import Path
PKG = {'vera/__init__.py': '', 'vera/analysis.py': '"""Claim table, scores, and evaluation.\n\nProtocol (fixed before seeing test results):\n  * Primary set: generated claims that assert an abnormal finding (not negated,\n    not hedged), are localizable, and get a 0/1 label from the reference.\n  * Risk = -alignment (low attention on the named region -> likely unsupported).\n  * Every choice (layer, head subset, threshold) is made on the CALIBRATION\n    split only; the TEST split is touched once for reporting.\n  * CIs: 1000x bootstrap over images (claims from one image move together).\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .anatomy import rect_masks, union\nfrom .labels import label_claim, reference_index\nfrom .text import extract_claims\n\nFLAG_RATES = (0.1, 0.2, 0.3, 0.5)\n\n\n# ---------------------------------------------------------------------------\n# Building the claim table\n# ---------------------------------------------------------------------------\ndef build_table(manifest, attn_dir, seg_dir=None, log=print):\n    attn_dir = Path(attn_dir)\n    seg_dir = Path(seg_dir) if seg_dir else None\n    rows, Pl, Ph, Rm, VM, CF, Mr, Ms = [], [], [], [], [], [], [], []\n    rect_cache = {}\n    n_img = 0\n    for it in manifest:\n        mj, nz = attn_dir / f"{it[\'image_id\']}.json", attn_dir / f"{it[\'image_id\']}.npz"\n        if not (mj.exists() and nz.exists()):\n            continue\n        meta, arr = json.loads(mj.read_text()), np.load(nz)\n        G = meta["grid"]\n        if G not in rect_cache:\n            rect_cache[G] = rect_masks(G)\n        rect = rect_cache[G]\n        seg = None\n        if seg_dir is not None and (seg_dir / f"{it[\'image_id\']}.npz").exists():\n            sz = np.load(seg_dir / f"{it[\'image_id\']}.npz")\n            if bool(sz["valid"]):\n                seg = {k: sz[k] for k in sz.files if k != "valid"}\n        ref = reference_index(it.get("findings", ""), it.get("impression", ""),\n                              it.get("mesh", ""), it.get("problems", ""))\n        span2s = {(a, b): i for i, (_, a, b) in enumerate(meta["sentences"])}\n        n_img += 1\n        lm = arr["layer_map"].astype(np.float32)\n        hp = arr["head_pool"].astype(np.float32)\n        rm = arr["report_map"].astype(np.float32)\n        for c in extract_claims(meta["report"]):\n            si = span2s.get((c.sent_start, c.sent_end))\n            if si is None or not c.localizable:\n                continue\n            d = c.to_dict()\n            d.update(label_claim(d, ref))\n            d.update(image_id=it["image_id"], split=it["split"], ref_abnormal=it.get("ref_abnormal"),\n                     seg_valid=seg is not None, sent_index=si)\n            mr = union(rect, c.zones)\n            ms = union(seg, c.zones) if seg is not None else None\n            if mr is None:\n                continue\n            if ms is None or ms.sum() == 0:\n                ms = mr\n                d["seg_used"] = False\n            else:\n                d["seg_used"] = True\n            L = lm.shape[1]\n            p = lm[si].reshape(L, -1)\n            Pl.append((p / np.clip(p.sum(1, keepdims=True), 1e-12, None)).astype(np.float16))\n            h = hp[si].reshape(hp.shape[1], hp.shape[2], -1)\n            Ph.append((h / np.clip(h.sum(-1, keepdims=True), 1e-12, None)).astype(np.float16))\n            r = rm.reshape(L, -1)\n            Rm.append((r / np.clip(r.sum(1, keepdims=True), 1e-12, None)).astype(np.float16))\n            VM.append(arr["vis_mass"][si])\n            CF.append(arr["conf"][si])\n            Mr.append(mr.ravel())\n            Ms.append(ms.ravel())\n            rows.append(d)\n    log(f"images: {n_img}, localizable claims: {len(rows)}")\n    A = dict(Pl=np.stack(Pl).astype(np.float16), Ph=np.stack(Ph).astype(np.float16),\n             Rm=np.stack(Rm).astype(np.float16), VM=np.stack(VM), CF=np.stack(CF),\n             Mr=np.stack(Mr), Ms=np.stack(Ms)) if rows else {}\n    return rows, A\n\n\n# ---------------------------------------------------------------------------\n# Metrics\n# ---------------------------------------------------------------------------\ndef auroc(y, s):\n    y, s = np.asarray(y), np.asarray(s, dtype=float)\n    ok = ~np.isnan(s)\n    y, s = y[ok], s[ok]\n    n1, n0 = int(y.sum()), int((1 - y).sum())\n    if n1 == 0 or n0 == 0:\n        return float("nan")\n    order = np.argsort(s, kind="mergesort")\n    ranks = np.empty(len(s))\n    sv = s[order]\n    i = 0\n    while i < len(s):  # average ranks for ties\n        j = i\n        while j + 1 < len(s) and sv[j + 1] == sv[i]:\n            j += 1\n        ranks[order[i:j + 1]] = (i + j) / 2 + 1\n        i = j + 1\n    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))\n\n\ndef auprc(y, s):\n    y, s = np.asarray(y), np.asarray(s, dtype=float)\n    o = np.argsort(-s)\n    y = y[o]\n    tp = np.cumsum(y)\n    prec = tp / np.arange(1, len(y) + 1)\n    return float((prec * y).sum() / max(1, y.sum()))\n\n\ndef boot_ci(y, s, groups, fn=auroc, n=1000, seed=0):\n    rng = np.random.default_rng(seed)\n    y, s, groups = np.asarray(y), np.asarray(s, float), np.asarray(groups)\n    ug = np.unique(groups)\n    idx_by = {g: np.nonzero(groups == g)[0] for g in ug}\n    vals = []\n    for _ in range(n):\n        pick = rng.choice(ug, len(ug))\n        idx = np.concatenate([idx_by[g] for g in pick])\n        v = fn(y[idx], s[idx])\n        if not np.isnan(v):\n            vals.append(v)\n    if not vals:\n        return [float("nan"), float("nan")]\n    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))]\n\n\ndef at_flag_rate(y_te, r_te, r_cal, rate):\n    thr = np.quantile(r_cal, 1 - rate)\n    f = r_te >= thr\n    tp = int((f & (y_te == 1)).sum())\n    prec = tp / max(1, int(f.sum()))\n    rec = tp / max(1, int(y_te.sum()))\n    f1 = 2 * prec * rec / max(1e-12, prec + rec)\n    return dict(target_rate=rate, test_flag_rate=float(f.mean()), precision=prec, recall=rec, f1=f1)\n\n\n# ---------------------------------------------------------------------------\n# Scores\n# ---------------------------------------------------------------------------\ndef region_scores(P, M, prior=None):\n    """P: [N, ..., K] normalised maps; M: [N, K] soft masks -> raw, lift, excess [N, ...]."""\n    shp = (M.shape[0],) + (1,) * (P.ndim - 2) + (M.shape[1],)\n    Mb = M.reshape(shp)\n    raw = (P * Mb).sum(-1)\n    area = M.mean(1).reshape((M.shape[0],) + (1,) * (P.ndim - 2))\n    out = dict(raw=raw, lift=raw / np.clip(area, 1e-6, None))\n    if prior is not None:\n        out["excess"] = raw - (prior[None] * Mb).sum(-1)\n    return out\n\n\ndef region_scores_chunked(P, M, prior=None, step=512):\n    """region_scores in row chunks so float32 temporaries stay small."""\n    outs = [region_scores(P[i:i + step].astype(np.float32), M[i:i + step], prior)\n            for i in range(0, len(P), step)]\n    return {k: np.concatenate([o[k] for o in outs]) for k in outs[0]}\n\n\ndef fit_probe(X, y, cal, te, groups, seed=0):\n    """L2 logistic regression trained on calibration claims only; C picked by grouped CV\n    on calibration. Returns risk for every row (test rows are never seen in fitting)."""\n    from sklearn.linear_model import LogisticRegression\n    from sklearn.model_selection import GridSearchCV, GroupKFold\n    from sklearn.pipeline import make_pipeline\n    from sklearn.preprocessing import StandardScaler\n    X = np.nan_to_num(X.reshape(len(X), -1).astype(np.float32))\n    k = max(2, min(5, len(np.unique(groups[cal]))))\n    cv = list(GroupKFold(k).split(X[cal], y[cal], groups[cal]))\n    gs = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),\n                      {"logisticregression__C": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]},\n                      cv=cv, scoring="roc_auc")\n    gs.fit(X[cal], y[cal])\n    return gs.predict_proba(X)[:, 1], float(gs.best_score_), gs.best_params_["logisticregression__C"]\n\n\ndef pool_mask(M, G, k=4):\n    g = G // k\n    return M.reshape(-1, g, k, g, k).mean((2, 4)).reshape(len(M), -1)\n\n\ndef entropy(P):\n    P = np.clip(P, 1e-12, None)\n    return -(P * np.log2(P)).sum(-1)\n\n\n# ---------------------------------------------------------------------------\n# Evaluation\n# ---------------------------------------------------------------------------\ndef evaluate(rows, A, out_dir, n_boot=1000, top_k_heads=8, log=print):\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    N = len(rows)\n    lab = np.array([np.nan if r["label"] is None else r["label"] for r in rows], float)\n    split = np.array([r["split"] for r in rows])\n    imgs = np.array([r["image_id"] for r in rows])\n    kind = np.array([r["kind"] for r in rows])\n    neg = np.array([r["negated"] for r in rows])\n    unc = np.array([r["uncertain"] for r in rows])\n    tier = np.array([r["tier"] for r in rows])\n\n    prim = (kind == "abnormal") & ~neg & ~unc & ~np.isnan(lab)\n    cal, te = prim & (split == "calib"), prim & (split == "test")\n    y = np.nan_to_num(lab).astype(int)\n    Pl = A["Pl"]                              # [N, L, G*G] float16\n    Ph = A["Ph"]                              # [N, L, H, g*g] float16\n    G = int(round(np.sqrt(Pl.shape[-1])))\n    L, H = Ph.shape[1], Ph.shape[2]\n\n    # priors from calibration sentences only (any claim kind), per layer / head\n    cal_any = split == "calib"\n    prior_l = Pl[cal_any].astype(np.float32).mean(0)   # [L, K]\n    prior_h = Ph[cal_any].astype(np.float32).mean(0)   # [L, H, k]\n\n    feats = {}\n    for mname, M in (("seg", A["Ms"]), ("rect", A["Mr"])):\n        s = region_scores_chunked(Pl, M, prior_l)       # each [N, L]\n        for met, v in s.items():\n            feats[f"{mname}:{met}"] = v\n        mp = pool_mask(M, G)\n        sh = region_scores_chunked(Ph, mp, prior_h)     # [N, L, H]\n        for met, v in sh.items():\n            feats[f"{mname}:{met}:head"] = v\n\n    results = dict(n_claims_total=N, n_primary_calib=int(cal.sum()), n_primary_test=int(te.sum()),\n                   prevalence_calib=float(y[cal].mean()) if cal.any() else None,\n                   prevalence_test=float(y[te].mean()) if te.any() else None,\n                   n_images_test=int(len(np.unique(imgs[te]))), grid=G, n_layers=L, n_heads=H,\n                   seg_used_fraction=float(np.mean([r.get("seg_used", False) for r, p in zip(rows, prim) if p])) if prim.any() else None)\n\n    def report(name, risk, mask=te, cal_mask=cal):\n        r_te, y_te = risk[mask], y[mask]\n        rec = dict(name=name, auroc=auroc(y_te, r_te), auroc_ci=boot_ci(y_te, r_te, imgs[mask], n=n_boot),\n                   auprc=auprc(y_te, r_te), n=int(mask.sum()))\n        if cal_mask is not None and cal_mask.any():\n            rec["calib_auroc"] = auroc(y[cal_mask], risk[cal_mask])\n            rec["at_flag_rate"] = [at_flag_rate(y_te, r_te, risk[cal_mask], fr) for fr in FLAG_RATES]\n        return rec\n\n    # ---- layer selection on calibration (pre-specified direction: risk = -score)\n    layer_curves, selected = {}, {}\n    for key in ["seg:raw", "seg:lift", "seg:excess", "rect:raw", "rect:lift", "rect:excess"]:\n        v = feats[key]\n        cal_curve = [auroc(y[cal], -v[cal, l]) for l in range(L)]\n        test_curve = [auroc(y[te], -v[te, l]) for l in range(L)]\n        layer_curves[key] = dict(calib=cal_curve, test=test_curve)\n        lbest = int(np.nanargmax(cal_curve))\n        selected[key] = lbest\n    results["layer_curves"] = layer_curves\n\n    primary_key = "seg:lift"\n    lsel = selected[primary_key]\n    risk_primary = -feats[primary_key][:, lsel]\n    results["primary"] = dict(config=f"{primary_key}, layer {lsel} (selected on calibration)",\n                              **report("VERA (seg, lift, calib-selected layer)", risk_primary))\n\n    # ---- head ensemble: top-k heads by calibration AUROC, averaged z-scores\n    hk = feats[f"{primary_key}:head"]                                   # [N, L, H]\n    flat = hk.reshape(N, -1)\n    cal_h = np.array([auroc(y[cal], -flat[cal, j]) for j in range(flat.shape[1])])\n    top = np.argsort(-np.nan_to_num(cal_h, nan=0))[:top_k_heads]\n    mu, sd = flat[cal][:, top].mean(0), flat[cal][:, top].std(0) + 1e-9\n    risk_heads = -((flat[:, top] - mu) / sd).mean(1)\n    results["head_ensemble"] = dict(heads=[(int(j // H), int(j % H), float(cal_h[j])) for j in top],\n                                    **report(f"VERA head ensemble (top {top_k_heads}, calib-selected)", risk_heads))\n\n    # ---- calibration-trained probes (all fitting on calib claims; test touched once)\n    # region: alignment of every head with the claim\'s own region.\n    # shuffled: same features, but each claim gets another claim\'s region -> tests whether\n    #           the NAMED region carries the signal (grounding) or just generic attention.\n    # share: per-head attention share to the image, no region at all (non-spatial control).\n    CF, VM = A["CF"], A["VM"]\n    Ms_pool = pool_mask(A["Ms"], G)\n    perm = np.random.default_rng(1).permutation(N)\n    sh_shuf = region_scores_chunked(Ph, Ms_pool[perm], prior_h)\n    X_region = np.concatenate([feats["seg:raw:head"].reshape(N, -1), feats["seg:lift:head"].reshape(N, -1),\n                               feats["seg:excess:head"].reshape(N, -1)], 1)\n    X_shuf = np.concatenate([sh_shuf["raw"].reshape(N, -1), sh_shuf["lift"].reshape(N, -1),\n                             sh_shuf["excess"].reshape(N, -1)], 1)\n    X_share = np.concatenate([VM.reshape(N, -1), CF], 1)\n    probes = []\n    for nm, X in (("Probe: region alignment, all heads (VERA-probe)", X_region),\n                  ("Probe: region alignment with SHUFFLED regions (control)", X_shuf),\n                  ("Probe: image-attention share + confidence, no region (control)", X_share),\n                  ("Probe: region alignment + share + confidence", np.concatenate([X_region, X_share], 1))):\n        if cal.sum() < 20:\n            break\n        risk_p, cv_auc, C = fit_probe(X, y, cal, te, imgs)\n        rec = report(nm, risk_p)\n        rec.update(calib_cv_auroc=cv_auc, C=C)\n        probes.append(rec)\n        if nm.startswith("Probe: region alignment, all heads"):\n            risk_probe = risk_p\n    results["probes"] = probes\n\n    # ---- baselines on the same claims\n    lay = lsel\n    base = [\n        report("Confidence: mean token log-prob", -CF[:, 0]),\n        report("Confidence: min token log-prob", -CF[:, 1]),\n        report("Visual reliance (attention share to image, all layers/heads)", -VM.mean((1, 2))),\n        report(f"Attention entropy (layer {lay})", entropy(Pl[:, lay].astype(np.float32))),\n    ]\n    # v1 recipe at the corrected grid: rect masks, raw score, last 4 layers, report-level map\n    Rm = A["Rm"].astype(np.float32)\n    rep_raw = region_scores(Rm[:, -4:].mean(1, keepdims=True), A["Mr"])["raw"][:, 0]\n    base.append(report("v1 recipe (rect, raw, last-4 layers, whole-report map)", -rep_raw))\n    rng = np.random.default_rng(0)\n    base.append(report("Random", rng.random(N)))\n    results["baselines"] = base\n    results["flag_all_f1_test"] = float(2 * y[te].mean() / (1 + y[te].mean())) if te.any() else None\n\n    # ---- ablations (test AUROC; layer chosen on calibration for each variant)\n    abl = []\n    for key in ["seg:raw", "seg:lift", "seg:excess", "rect:raw", "rect:lift", "rect:excess"]:\n        abl.append(report(f"{key} @ calib layer {selected[key]}", -feats[key][:, selected[key]]))\n    last4 = region_scores(Pl[:, -4:].mean(1, keepdims=True), A["Ms"])["lift"][:, 0]\n    abl.append(report("seg:lift, last-4-layer average (v1 layer choice)", -last4))\n    rep_lift = region_scores(Rm[:, lsel:lsel + 1], A["Ms"])["lift"][:, 0]\n    abl.append(report(f"seg:lift, whole-report map instead of sentence (layer {lsel})", -rep_lift))\n    results["ablations"] = abl\n\n    # ---- per tier, and secondary claim types\n    results["per_tier"] = {t: report(f"primary, tier={t}", risk_primary, mask=te & (tier == t), cal_mask=None)\n                           for t in ("critical", "moderate", "mild")}\n    sec = {}\n    lv = ~np.isnan(lab)\n    for nm, m in (("negated_claims", (kind == "abnormal") & neg & lv),\n                  ("normal_claims", (kind == "normal") & lv),\n                  ("hedged_claims", (kind == "abnormal") & ~neg & unc & lv)):\n        sec[nm] = report(nm, risk_primary, mask=m & (split == "test"), cal_mask=None)\n    results["secondary"] = sec\n    results["label_reasons_primary"] = {k: int(v) for k, v in zip(*np.unique(\n        [r["label_reason"] for r, p in zip(rows, prim) if p], return_counts=True))} if prim.any() else {}\n\n    _figures(results, feats, y, te, risk_primary, base, out_dir,\n             risk_probe=risk_probe if probes else None)\n    (out_dir / "results.json").write_text(json.dumps(results, indent=1, default=float))\n    _write_claims_csv(rows, risk_primary, out_dir / "claims.csv")\n    (out_dir / "summary.md").write_text(summary_md(results))\n    log(summary_md(results))\n    return results, risk_primary\n\n\ndef _write_claims_csv(rows, risk, path):\n    import csv\n    keys = ["image_id", "split", "sentence", "kind", "finding", "tier", "negated", "uncertain",\n            "relational", "location_text", "zones", "label", "label_reason", "seg_used"]\n    with open(path, "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(keys + ["vera_risk"])\n        for r, s in zip(rows, risk):\n            w.writerow([("|".join(r[k]) if k == "zones" else r.get(k)) for k in keys] + [float(s)])\n\n\ndef _fmt(r):\n    ci = r.get("auroc_ci", [np.nan, np.nan])\n    return f"{r[\'auroc\']:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"\n\n\ndef summary_md(R):\n    lines = ["# VERA v3 results", "",\n             f"- Primary claims: calib {R[\'n_primary_calib\']}, test {R[\'n_primary_test\']} "\n             f"({R[\'n_images_test\']} test images); hallucination prevalence test = {R[\'prevalence_test\']}",\n             f"- Visual grid {R[\'grid\']}x{R[\'grid\']}, {R[\'n_layers\']} layers x {R[\'n_heads\']} heads; "\n             f"segmentation masks used for {R[\'seg_used_fraction\']} of primary claims",\n             f"- Flag-all F1 on test: {R[\'flag_all_f1_test\']}", "",\n             "## Primary (pre-specified)", f"- {R[\'primary\'][\'config\']}: AUROC {_fmt(R[\'primary\'])}, "\n             f"calib AUROC {R[\'primary\'].get(\'calib_auroc\', float(\'nan\')):.3f}",\n             f"- Head ensemble: AUROC {_fmt(R[\'head_ensemble\'])}", "",\n             "## Calibration-trained probes (test AUROC, 95% CI)"]\n    lines += [f"- {p[\'name\']}: {_fmt(p)} (calib CV {p[\'calib_cv_auroc\']:.3f}, C={p[\'C\']})" for p in R.get("probes", [])]\n    lines += ["", "## Baselines (test AUROC, 95% CI)"]\n    lines += [f"- {b[\'name\']}: {_fmt(b)}" for b in R["baselines"]]\n    lines += ["", "## Ablations (test AUROC, 95% CI)"] + [f"- {a[\'name\']}: {_fmt(a)}" for a in R["ablations"]]\n    lines += ["", "## Per tier"] + [f"- {k}: {_fmt(v)} (n={v[\'n\']})" for k, v in R["per_tier"].items()]\n    lines += ["", "## Operating points (primary, thresholds from calibration)"]\n    for op in R["primary"].get("at_flag_rate", []):\n        lines.append(f"- target flag rate {op[\'target_rate\']:.0%}: test flag rate {op[\'test_flag_rate\']:.1%}, "\n                     f"recall {op[\'recall\']:.3f}, precision {op[\'precision\']:.3f}, F1 {op[\'f1\']:.3f}")\n    lines += ["", "## Secondary claim types (same score, test)"] + [f"- {k}: {_fmt(v)} (n={v[\'n\']})" for k, v in R["secondary"].items()]\n    lines += ["", f"Primary label reasons: {R[\'label_reasons_primary\']}"]\n    return "\\n".join(lines)\n\n\ndef _figures(R, feats, y, te, risk_primary, base, out_dir, risk_probe=None):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(1, 2, figsize=(12, 4))\n    for key, c in [("seg:lift", "C0"), ("seg:excess", "C1"), ("rect:lift", "C2")]:\n        cur = R["layer_curves"][key]\n        ax[0].plot(cur["calib"], color=c, label=f"{key} calib")\n        ax[0].plot(cur["test"], color=c, ls="--", alpha=.6, label=f"{key} test")\n    ax[0].axhline(.5, color="k", lw=.8)\n    ax[0].set_xlabel("decoder layer")\n    ax[0].set_ylabel("AUROC")\n    ax[0].legend(fontsize=7)\n    ax[0].set_title("AUROC by layer")\n\n    def roc(yv, s):\n        o = np.argsort(-s)\n        yv = yv[o]\n        tpr = np.cumsum(yv) / max(1, yv.sum())\n        fpr = np.cumsum(1 - yv) / max(1, (1 - yv).sum())\n        return np.r_[0, fpr], np.r_[0, tpr]\n\n    fx, fy = roc(y[te], risk_primary[te])\n    ax[1].plot(fx, fy, lw=2, label=f"VERA training-free ({R[\'primary\'][\'auroc\']:.3f})")\n    if risk_probe is not None:\n        px, py = roc(y[te], risk_probe[te])\n        ax[1].plot(px, py, lw=2, label=f"VERA-probe ({R[\'probes\'][0][\'auroc\']:.3f})")\n    ax[1].plot([0, 1], [0, 1], "k:", lw=.8)\n    ax[1].set_xlabel("FPR")\n    ax[1].set_ylabel("TPR")\n    ax[1].legend(fontsize=8)\n    ax[1].set_title("Test ROC (primary)")\n    fig.tight_layout()\n    fig.savefig(out_dir / "fig_layers_roc.png", dpi=160)\n    plt.close(fig)\n', 'vera/anatomy.py': '"""Anatomical zone masks on the visual-token grid.\n\nTwo atlases, both returned as soft masks on a G x G grid (G = 32 for CheXagent-2-3b):\n  * "rect": the v1 hand-drawn rectangles (normalized image coords), kept for ablation.\n  * "seg":  patient-specific zones derived from torchxrayvision\'s chest anatomy\n            segmentation (PSPNet, ChestX-Det targets). This answers the reviewer\n            concern that the rectangles were never validated.\n\nConvention: radiological view, patient RIGHT is on the IMAGE LEFT (x < 0.5).\nCheXagent resizes the full image to a square without cropping, so normalized\nimage coordinates map directly onto the token grid.\n"""\nfrom __future__ import annotations\n\nfrom typing import Dict, Optional\n\nimport numpy as np\n\nRECT_ZONES = {  # (x_min, y_min, x_max, y_max)\n    "right_upper_lobe":         (0.08, 0.08, 0.42, 0.38),\n    "right_middle_lobe":        (0.12, 0.33, 0.42, 0.55),\n    "right_lower_lobe":         (0.12, 0.48, 0.48, 0.82),\n    "left_upper_lobe":          (0.58, 0.08, 0.92, 0.38),\n    "left_middle_zone":         (0.55, 0.33, 0.88, 0.55),\n    "left_lower_lobe":          (0.52, 0.48, 0.88, 0.82),\n    "right_apex":               (0.10, 0.05, 0.42, 0.22),\n    "left_apex":                (0.58, 0.05, 0.90, 0.22),\n    "mediastinum":              (0.35, 0.05, 0.65, 0.65),\n    "cardiac_silhouette":       (0.28, 0.35, 0.72, 0.78),\n    "right_hilum":              (0.33, 0.22, 0.48, 0.48),\n    "left_hilum":               (0.52, 0.22, 0.67, 0.48),\n    "right_costophrenic_angle": (0.08, 0.72, 0.32, 0.95),\n    "left_costophrenic_angle":  (0.68, 0.72, 0.92, 0.95),\n    "right_hemidiaphragm":      (0.08, 0.68, 0.48, 0.88),\n    "left_hemidiaphragm":       (0.52, 0.68, 0.92, 0.88),\n}\nZONE_NAMES = list(RECT_ZONES)\n\n\ndef rect_masks(G: int) -> Dict[str, np.ndarray]:\n    """Soft (fractional-coverage) rectangle masks on a G x G grid."""\n    out = {}\n    edges = np.linspace(0, 1, G + 1)\n    for z, (x0, y0, x1, y1) in RECT_ZONES.items():\n        cx = np.clip(np.minimum(edges[1:], x1) - np.maximum(edges[:-1], x0), 0, None) * G\n        cy = np.clip(np.minimum(edges[1:], y1) - np.maximum(edges[:-1], y0), 0, None) * G\n        out[z] = np.outer(cy, cx).astype(np.float32)\n    return out\n\n\n# --------------------------------------------------------------------------\n# Segmentation-derived zones\n# --------------------------------------------------------------------------\n_SEG_MODEL = None\n\n\ndef _load_seg(device):\n    global _SEG_MODEL\n    if _SEG_MODEL is None:\n        import torchxrayvision as xrv\n        _SEG_MODEL = xrv.baseline_models.chestx_det.PSPNet().to(device).eval()\n    return _SEG_MODEL\n\n\ndef segment(image_path: str, device="cpu") -> Dict[str, np.ndarray]:\n    """Run chest anatomy segmentation; returns {target: bool mask 512x512} on the\n    full image resized to a square (same geometry CheXagent sees)."""\n    import torch\n    import torch.nn.functional as F\n    import torchxrayvision as xrv\n    from PIL import Image\n\n    img = np.array(Image.open(image_path).convert("L")).astype(np.float32)\n    img = xrv.datasets.normalize(img, 255)\n    x = torch.from_numpy(img)[None, None].to(device)\n    x = F.interpolate(x, size=(512, 512), mode="bilinear", align_corners=False)\n    model = _load_seg(device)\n    with torch.no_grad():\n        y = torch.sigmoid(model(x))[0].cpu().numpy() > 0.5\n    return {t: y[i] for i, t in enumerate(model.targets)}\n\n\ndef _centroid_x(m: np.ndarray) -> float:\n    ys, xs = np.nonzero(m)\n    return float(xs.mean()) / m.shape[1] if len(xs) else 0.5\n\n\ndef _rows_band(m: np.ndarray, lo: float, hi: float) -> np.ndarray:\n    """Part of mask m between fractions lo..hi of ITS OWN vertical extent."""\n    ys = np.nonzero(m.any(1))[0]\n    if len(ys) == 0:\n        return np.zeros_like(m)\n    top, bot = ys[0], ys[-1] + 1\n    a, b = int(top + lo * (bot - top)), int(top + hi * (bot - top))\n    out = np.zeros_like(m)\n    out[a:b] = m[a:b]\n    return out\n\n\ndef _lateral_half(m: np.ndarray, patient_side: str) -> np.ndarray:\n    xs = np.nonzero(m.any(0))[0]\n    if len(xs) == 0:\n        return m\n    mid = (xs[0] + xs[-1] + 1) // 2\n    out = m.copy()\n    if patient_side == "right":   # image left -> lateral = small x\n        out[:, mid:] = False\n    else:\n        out[:, :mid] = False\n    return out\n\n\ndef seg_zone_masks(seg: Dict[str, np.ndarray], G: int) -> Optional[Dict[str, np.ndarray]]:\n    """Derive the 16 zones from segmentation; None if it does not look frontal."""\n    lungs = [seg["Left Lung"], seg["Right Lung"]]\n    lungs = [m for m in lungs if m.mean() > 0.02]\n    if len(lungs) != 2:\n        return None\n    lungs.sort(key=_centroid_x)              # image-left first = patient right\n    R, L = lungs\n    if not (_centroid_x(R) < 0.5 < _centroid_x(L)) or seg["Heart"].mean() < 0.01:\n        return None\n    hil = [seg["Left Hilus Pulmonis"], seg["Right Hilus Pulmonis"]]\n    hil_r = np.logical_or.reduce([h for h in hil if h.any() and _centroid_x(h) < 0.5] or [np.zeros_like(R)])\n    hil_l = np.logical_or.reduce([h for h in hil if h.any() and _centroid_x(h) >= 0.5] or [np.zeros_like(R)])\n    dia = seg["Facies Diaphragmatica"]\n    half = dia.shape[1] // 2\n    dia_r, dia_l = dia.copy(), dia.copy()\n    dia_r[:, half:] = False\n    dia_l[:, :half] = False\n    full = {\n        "right_upper_lobe": _rows_band(R, 0, 1 / 3),\n        "right_middle_lobe": _rows_band(R, 1 / 3, 2 / 3),\n        "right_lower_lobe": _rows_band(R, 2 / 3, 1),\n        "left_upper_lobe": _rows_band(L, 0, 1 / 3),\n        "left_middle_zone": _rows_band(L, 1 / 3, 2 / 3),\n        "left_lower_lobe": _rows_band(L, 2 / 3, 1),\n        "right_apex": _rows_band(R, 0, 0.2),\n        "left_apex": _rows_band(L, 0, 0.2),\n        "mediastinum": seg["Mediastinum"] | seg["Aorta"] | seg["Weasand"],\n        "cardiac_silhouette": seg["Heart"],\n        "right_hilum": hil_r,\n        "left_hilum": hil_l,\n        "right_costophrenic_angle": _lateral_half(_rows_band(R, 0.8, 1), "right"),\n        "left_costophrenic_angle": _lateral_half(_rows_band(L, 0.8, 1), "left"),\n        "right_hemidiaphragm": dia_r,\n        "left_hemidiaphragm": dia_l,\n    }\n    return {z: downsample(m, G) for z, m in full.items()}\n\n\ndef downsample(m: np.ndarray, G: int) -> np.ndarray:\n    """Fraction of each grid cell covered by mask m (soft mask)."""\n    H = m.shape[0]\n    k = H // G\n    return m[:k * G, :k * G].reshape(G, k, G, k).mean((1, 3)).astype(np.float32)\n\n\ndef union(masks: Dict[str, np.ndarray], zones) -> Optional[np.ndarray]:\n    ms = [masks[z] for z in zones if z in masks]\n    if not ms:\n        return None\n    return np.maximum.reduce(ms)\n', 'vera/data.py': '"""IU CXR manifest: frontal views only, one image per study, split by study.\n\nFixes over v1: lateral views were mixed in (the frontal atlas cannot be valid on\nthem) and splits were by image, so two views of one study could land in\ndifferent splits. Here: projection == Frontal from indiana_projections.csv,\none frontal image per study uid, and the calibration/test split is by study.\n\nWe oversample studies whose reference report is abnormal so the evaluation has\nenough asserted findings to label. This is disclosed in the paper.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom pathlib import Path\nfrom typing import Dict, List\n\nKAGGLE_SLUG = "raddar/chest-xrays-indiana-university"\n\n\ndef download_iu(target: str | None = None) -> Path:\n    """Download via kagglehub. Set KAGGLE_USERNAME / KAGGLE_KEY as env vars (secrets)."""\n    import kagglehub\n    path = Path(kagglehub.dataset_download(KAGGLE_SLUG))\n    if target:\n        Path(target).mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef _find(root: Path, name: str) -> Path:\n    hits = list(root.rglob(name))\n    if not hits:\n        raise FileNotFoundError(f"{name} not found under {root}")\n    return hits[0]\n\n\ndef build_manifest(iu_root: str | Path, n_total: int = 3000, abnormal_frac: float = 0.75,\n                   calib_frac: float = 0.4, seed: int = 42) -> List[Dict]:\n    """n_total frontal studies, ~abnormal_frac with an abnormal reference; any shortfall is\n    filled from the other group. calib_frac is larger than v2 because the probes train on it."""\n    import pandas as pd\n\n    root = Path(iu_root)\n    reports = pd.read_csv(_find(root, "indiana_reports.csv"))\n    proj = pd.read_csv(_find(root, "indiana_projections.csv"))\n    img_dir = _find(root, "images_normalized") if list(root.rglob("images_normalized")) else root\n    files = {p.name: p for p in img_dir.rglob("*.png")}\n\n    frontal = proj[proj["projection"].str.lower() == "frontal"]\n    first_frontal = frontal.groupby("uid")["filename"].first().to_dict()\n\n    rows = []\n    for _, r in reports.iterrows():\n        uid = int(r["uid"])\n        fn = first_frontal.get(uid)\n        if fn is None or fn not in files:\n            continue\n        findings = "" if pd.isna(r.get("findings")) else str(r["findings"])\n        impression = "" if pd.isna(r.get("impression")) else str(r["impression"])\n        if not (findings.strip() or impression.strip()):\n            continue\n        problems = "" if pd.isna(r.get("Problems")) else str(r["Problems"])\n        mesh = "" if pd.isna(r.get("MeSH")) else str(r["MeSH"])\n        abnormal = problems.strip().lower() not in ("normal", "")\n        rows.append(dict(image_id=fn.replace(".png", ""), uid=uid, image_path=str(files[fn]),\n                         findings=findings, impression=impression, problems=problems,\n                         mesh=mesh, ref_abnormal=abnormal))\n\n    rng = random.Random(seed)\n    ab = [x for x in rows if x["ref_abnormal"]]\n    no = [x for x in rows if not x["ref_abnormal"]]\n    rng.shuffle(ab)\n    rng.shuffle(no)\n    n_ab = min(len(ab), int(round(abnormal_frac * n_total)))\n    n_no = min(len(no), n_total - n_ab)\n    n_ab = min(len(ab), n_total - n_no)\n    pick = ab[:n_ab] + no[:n_no]\n    rng.shuffle(pick)\n    n_cal = int(round(calib_frac * len(pick)))\n    for i, x in enumerate(pick):\n        x["split"] = "calib" if i < n_cal else "test"\n    return pick\n\n\ndef save_manifest(items, path):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(items, indent=1))\n\n\ndef load_manifest(path):\n    return json.loads(Path(path).read_text())\n', 'vera/extract.py': '"""Generate a report and store sentence-level attention to the image tokens.\n\nKey fixes over v1:\n  * CheXagent-2-3b inserts 1024 visual tokens (32 x 32 grid; the image is\n    resized to 512 px and SigLIP uses 16 px patches). v1 kept the first 576\n    tokens and reshaped them to 24 x 24, which scrambled the spatial layout and\n    dropped the bottom 44% of the image. Here the grid size is read from the\n    actual image span in the input ids and asserted to be square.\n  * Token -> character offsets come from incremental decoding, so each sentence\n    gets exactly its own tokens (v1 used a proportional char->token guess).\n  * We keep every layer, and every head (pooled to 8 x 8), plus the\n    pre-renormalisation share of attention that went to the image at all.\n\nPer image we save  <out>/<image_id>.npz  and  <out>/<image_id>.json\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .text import split_sentences\n\nPROMPT = (\n    "You are an expert radiologist. Generate a detailed radiology report "\n    "for this chest X-ray image. Include findings about the lungs, heart, "\n    "mediastinum, and any abnormalities. Structure your response with "\n    "Findings and Impression sections."\n)\nHEAD_POOL = 4  # 32x32 -> 8x8 for per-head maps (storage)\n\n\ndef _build_inputs(tok, image_path: str, prompt: str, device):\n    query = tok.from_list_format([{"image": image_path}, {"text": prompt}])\n    conv = [{"from": "system", "value": "You are a helpful assistant."},\n            {"from": "human", "value": query}]\n    text = tok.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)\n    enc = tok(text=text, return_tensors="pt").to(device)\n    ids = enc.input_ids[0].tolist()\n    a = ids.index(tok.img_start_id)\n    b = ids.index(tok.img_end_id)\n    n_vis = b - a - 1\n    G = int(math.isqrt(n_vis))\n    if G * G != n_vis:\n        raise ValueError(f"image span has {n_vis} tokens, not a square grid")\n    return enc, a + 1, b, G\n\n\ndef _token_char_ends(tok, gen_ids) -> list:\n    ends, prev = [], 0\n    for t in range(len(gen_ids)):\n        s = tok.decode(gen_ids[: t + 1], skip_special_tokens=True)\n        prev = max(prev, len(s))\n        ends.append(prev)\n    return ends\n\n\n@torch.no_grad()\ndef extract_one(model, tok, image_path: str, max_new_tokens: int = 256, prompt: str = PROMPT):\n    dev = next(model.parameters()).device\n    enc, v0, v1, G = _build_inputs(tok, image_path, prompt, dev)\n    L0 = enc.input_ids.shape[1]\n    t0 = time.time()\n    out = model.generate(\n        input_ids=enc.input_ids, attention_mask=enc.attention_mask,\n        max_new_tokens=max_new_tokens, do_sample=False,\n        output_attentions=True, output_scores=True, return_dict_in_generate=True,\n    )\n    gen = out.sequences[0, L0:]\n    T = min(len(gen), len(out.attentions))\n    n_layers = len(out.attentions[0])\n    n_heads = out.attentions[0][0].shape[1]\n    nv = v1 - v0\n\n    P = torch.empty((T, n_layers, n_heads, nv), dtype=torch.float16, device=dev)\n    lp = torch.empty(T, dtype=torch.float32)\n    for t in range(T):\n        rows = torch.stack([la[0, :, -1, :] for la in out.attentions[t]])  # [L,H,S]\n        P[t] = rows[:, :, v0:v1].half()\n        lp[t] = torch.log_softmax(out.scores[t][0].float(), -1)[gen[t]].item()\n    del out\n    torch.cuda.empty_cache()\n\n    gen_ids = gen[:T].tolist()\n    text = tok.decode(gen_ids, skip_special_tokens=True)\n    ends = _token_char_ends(tok, gen_ids)\n    starts = [0] + ends[:-1]\n\n    sents = split_sentences(text)\n    keep, tok_spans = [], []\n    for s in sents:\n        idx = [t for t in range(T) if starts[t] < s.end and ends[t] > s.start and ends[t] > starts[t]]\n        if idx:\n            keep.append(s)\n            tok_spans.append((idx[0], idx[-1] + 1))\n    S = len(keep)\n    g = G // HEAD_POOL\n    layer_map = np.zeros((S, n_layers, G, G), np.float16)\n    head_pool = np.zeros((S, n_layers, n_heads, g, g), np.float16)\n    vis_mass = np.zeros((S, n_layers, n_heads), np.float32)\n    conf = np.zeros((S, 2), np.float32)\n    for i, (ta, tb) in enumerate(tok_spans):\n        M = P[ta:tb].float().mean(0)                       # [L,H,nv]\n        vis_mass[i] = M.sum(-1).cpu().numpy()\n        layer_map[i] = M.mean(1).reshape(n_layers, G, G).cpu().numpy()\n        hp = M.reshape(n_layers, n_heads, g, HEAD_POOL, g, HEAD_POOL).sum((3, 5))\n        head_pool[i] = hp.cpu().numpy()\n        conf[i] = (lp[ta:tb].mean().item(), lp[ta:tb].min().item())\n    report_map = P.float().mean((0, 2)).reshape(n_layers, G, G).cpu().numpy().astype(np.float16)\n    del P\n\n    arrays = dict(layer_map=layer_map, head_pool=head_pool, vis_mass=vis_mass, conf=conf,\n                  report_map=report_map, token_logprob=lp.numpy())\n    meta = dict(image_path=image_path, report=text, grid=G, n_layers=n_layers, n_heads=n_heads,\n                n_tokens=T, sentences=[[s.text, s.start, s.end] for s in keep],\n                sentence_token_spans=tok_spans, seconds=round(time.time() - t0, 1))\n    return arrays, meta\n\n\ndef run_batch(model, tok, items, out_dir, max_new_tokens=256, log=print):\n    """items: list of dicts with image_id, image_path. Resumable."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    todo = [it for it in items if not (out_dir / f"{it[\'image_id\']}.json").exists()]\n    log(f"{len(items) - len(todo)} done already, {len(todo)} to go")\n    t_start, failures = time.time(), []\n    for k, it in enumerate(todo):\n        try:\n            try:\n                arrays, meta = extract_one(model, tok, it["image_path"], max_new_tokens)\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache()\n                arrays, meta = extract_one(model, tok, it["image_path"], 160)\n            np.savez_compressed(out_dir / f"{it[\'image_id\']}.npz", **arrays)\n            meta["image_id"] = it["image_id"]\n            (out_dir / f"{it[\'image_id\']}.json").write_text(json.dumps(meta))\n        except Exception as e:  # keep going, record failure\n            failures.append((it["image_id"], repr(e)[:300]))\n            torch.cuda.empty_cache()\n        if (k + 1) % 10 == 0 or k + 1 == len(todo):\n            el = time.time() - t_start\n            eta = el / (k + 1) * (len(todo) - k - 1)\n            log(f"[{k + 1}/{len(todo)}] {el / 60:.1f} min elapsed, ETA {eta / 60:.1f} min, failures {len(failures)}")\n    if failures:\n        (out_dir / "_failures.json").write_text(json.dumps(failures, indent=1))\n    return failures\n', 'vera/labels.py': '"""Finding-level, negation-aware labels against the radiologist reference.\n\nReplaces v1\'s templated NLI hypotheses ("There is no pleura in the cardiac.").\nFor each generated claim we ask a structured question of the reference report\n(findings + impression, plus IU\'s MeSH / Problems tags when available):\n\n  positive abnormal claim  F  -> 0 supported   if reference asserts F (side-compatible)\n                               1 hallucinated if reference negates F or never mentions it\n                               1 (mislocalized) if F is asserted only on the opposite side\n                               None            if the reference is only hedged about F\n  negated claim  "no F"     -> 1 if reference definitely asserts F, else 0\n  normal claim  (organ ok)  -> 1 if reference asserts an abnormality of that organ, else 0\n"""\nfrom __future__ import annotations\n\nimport re\nfrom typing import Dict, List, Optional\n\nfrom .text import _FINDING_RE, split_sentences, polarity\n\nORGAN_FINDINGS = {\n    "lungs": {"pneumothorax", "pleural_effusion", "mass", "pneumonia", "edema", "consolidation",\n              "opacity", "infiltrate", "atelectasis", "nodule", "vascular_congestion"},\n    "heart": {"cardiomegaly"},\n    "mediastinum": {"mediastinal_widening"},\n    "cardiomediastinal": {"cardiomegaly", "mediastinal_widening"},\n}\n_RIGHT = re.compile(r"\\bright\\b", re.I)\n_LEFT = re.compile(r"\\bleft\\b", re.I)\n_BOTH = re.compile(r"\\b(?:bilateral\\w*|bibas\\w*|both)\\b", re.I)\n\n\ndef _side_of(text: str) -> Optional[str]:\n    if _BOTH.search(text):\n        return "both"\n    r, l = bool(_RIGHT.search(text)), bool(_LEFT.search(text))\n    if r and l:\n        return "both"\n    return "right" if r else "left" if l else None\n\n\ndef reference_index(findings: str, impression: str, mesh: str = "", problems: str = "") -> Dict[str, dict]:\n    """status per canonical finding: positive / uncertain / negative (absent = missing key)."""\n    text = " ".join(x for x in [findings, impression] if x and x.lower() != "nan")\n    idx: Dict[str, dict] = {}\n\n    def put(name, status, side):\n        cur = idx.get(name)\n        rank = {"positive": 3, "uncertain": 2, "negative": 1}\n        if cur is None or rank[status] > rank[cur["status"]]:\n            idx[name] = {"status": status, "sides": set()}\n        if status == "positive" and side:\n            idx[name]["sides"].add(side)\n\n    for s in split_sentences(text):\n        for name, rx in _FINDING_RE.items():\n            for m in rx.finditer(s.text):\n                neg, unc = polarity(s.text, m.start(), m.end())\n                put(name, "negative" if neg else "uncertain" if unc else "positive", _side_of(s.text))\n    # MeSH / Problems tags, e.g. "Calcified Granuloma/lung/upper lobe/right; Cardiomegaly/mild"\n    for tag in re.split(r"[;|]", f"{mesh};{problems}"):\n        tag = tag.strip()\n        if not tag or tag.lower() in ("normal", "nan", "no indexing"):\n            continue\n        for name, rx in _FINDING_RE.items():\n            if rx.search(tag):\n                put(name, "positive", _side_of(tag))\n    return idx\n\n\ndef label_claim(claim: dict, ref: Dict[str, dict]) -> dict:\n    kind, f = claim["kind"], claim["finding"]\n    if kind == "normal":\n        bad = [x for x in ORGAN_FINDINGS.get(claim["location_text"], set())\n               if ref.get(x, {}).get("status") == "positive"]\n        return {"label": int(bool(bad)), "label_reason": "organ_abnormal_in_ref" if bad else "organ_normal_in_ref"}\n    r = ref.get(f)\n    if claim["negated"]:\n        hit = r is not None and r["status"] == "positive"\n        return {"label": int(hit), "label_reason": "ref_asserts_finding" if hit else "ref_consistent"}\n    if r is None:\n        return {"label": 1, "label_reason": "absent_in_ref"}\n    if r["status"] == "negative":\n        return {"label": 1, "label_reason": "negated_in_ref"}\n    if r["status"] == "uncertain":\n        return {"label": None, "label_reason": "hedged_in_ref"}\n    gen_side = _side_of(claim.get("location_text") or "") or _side_of(claim.get("sentence") or "")\n    sides = r["sides"]\n    if gen_side in ("right", "left") and sides and "both" not in sides and gen_side not in sides:\n        return {"label": 1, "label_reason": "mislocalized"}\n    return {"label": 0, "label_reason": "supported"}\n', 'vera/model.py': '"""Load CheXagent-2-3b for attention extraction.\n\nv1 loaded with device_map={"":0}, which left the SigLIP tower on the meta device;\nit then re-downloaded *base* XraySigLIP weights, which may not equal CheXagent\'s\ntrained vision encoder. Here we load fully on CPU first (no meta tensors), move to\nGPU, and report any checkpoint keys that failed to load.\n"""\nfrom __future__ import annotations\n\nimport os\n\nimport torch\n\nMODEL_ID = "StanfordAIMI/CheXagent-2-3b"\n\n\ndef load_chexagent(model_id: str = MODEL_ID, device: str = "cuda"):\n    import tokenizers\n    tokenizers.__version__ = "0.19.1"  # CheXagent remote code pins this string\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    token = os.environ.get("HF_TOKEN") or None\n    model, info = AutoModelForCausalLM.from_pretrained(\n        model_id,\n        torch_dtype=torch.float16,\n        trust_remote_code=True,\n        attn_implementation="eager",\n        low_cpu_mem_usage=False,      # no meta tensors\n        output_loading_info=True,\n        token=token,\n    )\n    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, token=token)\n\n    missing = [k for k in info.get("missing_keys", []) if "visual" in k]\n    unexpected = [k for k in info.get("unexpected_keys", []) if "visual" in k]\n    meta = [n for n, p in model.named_parameters() if p.device.type == "meta"]\n    if meta:\n        raise RuntimeError(f"{len(meta)} parameters still on meta device, e.g. {meta[:3]}")\n\n    # harmonise dtype (pos_embed is created in float32 by the remote code)\n    for p in model.parameters():\n        if p.dtype == torch.float32:\n            p.data = p.data.half()\n    for b in model.buffers():\n        if b.dtype == torch.float32:\n            b.data = b.data.half()\n\n    model = model.to(device).eval()\n    diag = {\n        "visual_missing_keys": len(missing),\n        "visual_unexpected_keys": len(unexpected),\n        "example_missing": missing[:5],\n        "n_layers": model.config.num_hidden_layers,\n        "n_heads": model.config.num_attention_heads,\n    }\n    return model, tok, diag\n', 'vera/pipeline.py': '"""Stage runner shared by the molab notebook and the CLI."""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport shutil\nimport zipfile\nfrom pathlib import Path\n\n\nclass Paths:\n    def __init__(self, work=None):\n        self.work = Path(work or os.environ.get("VERA_WORK", "vera_work")).resolve()\n        self.manifest = self.work / "manifest.json"\n        self.attn = self.work / "attention"\n        self.seg = self.work / "segmentation"\n        self.results = self.work / "results"\n        self.review = self.work / "results" / "review_pack"\n        for p in (self.work, self.attn, self.seg, self.results):\n            p.mkdir(parents=True, exist_ok=True)\n\n\ndef stage_data(P, n_total=3000, iu_root=None, log=print):\n    from .data import build_manifest, download_iu, save_manifest\n    root = iu_root or download_iu()\n    log(f"IU CXR at {root}")\n    items = build_manifest(root, n_total=n_total)\n    save_manifest(items, P.manifest)\n    n_cal = sum(x["split"] == "calib" for x in items)\n    log(f"manifest: {len(items)} frontal studies ({n_cal} calib / {len(items) - n_cal} test), "\n        f"{sum(x[\'ref_abnormal\'] for x in items)} with abnormal reference")\n    return items\n\n\ndef stage_extract(P, max_new_tokens=256, limit=None, log=print):\n    from .data import load_manifest\n    from .extract import run_batch\n    from .model import load_chexagent\n    items = load_manifest(P.manifest)[:limit]\n    model, tok, diag = load_chexagent()\n    log("model diagnostics:", diag)\n    (P.work / "model_diagnostics.json").write_text(json.dumps(diag, indent=1))\n    return run_batch(model, tok, items, P.attn, max_new_tokens, log=log)\n\n\ndef stage_segment(P, device="cuda", log=print):\n    from .data import load_manifest\n    from .segment import run_segmentation\n    items = load_manifest(P.manifest)\n    G = 32\n    metas = sorted(P.attn.glob("*.json"))\n    metas = [m for m in metas if not m.name.startswith("_")]\n    if metas:\n        G = json.loads(metas[0].read_text())["grid"]\n    run_segmentation(items, P.seg, G=G, device=device, log=log)\n\n\ndef stage_analyze(P, log=print):\n    from .analysis import build_table, evaluate\n    from .data import load_manifest\n    from .review import make_review_pack\n    items = load_manifest(P.manifest)\n    rows, A = build_table(items, P.attn, P.seg, log=log)\n    R, risk = evaluate(rows, A, P.results, log=log)\n    make_review_pack(rows, A, risk, P.review, P.attn)\n    return R\n\n\ndef stage_bundle(P, log=print):\n    """Zip only small outputs (results, review pack, diagnostics) to send back."""\n    out = P.work / "VERA_v3_results_for_claude.zip"\n    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:\n        for f in P.results.rglob("*"):\n            if f.is_file():\n                z.write(f, f.relative_to(P.work))\n        for f in [P.work / "model_diagnostics.json", P.attn / "_failures.json", P.manifest]:\n            if f.exists():\n                z.write(f, f.relative_to(P.work))\n        # 5 raw examples so the attention maps can be inspected directly\n        for m in sorted(P.attn.glob("*.json"))[:5]:\n            if not m.name.startswith("_"):\n                z.write(m, m.relative_to(P.work))\n                z.write(m.with_suffix(".npz"), m.with_suffix(".npz").relative_to(P.work))\n    log(f"bundle: {out} ({out.stat().st_size / 1e6:.1f} MB)")\n    return out\n', 'vera/review.py': '"""Radiologist review pack: overlays + a CSV sheet for a stratified sample."""\nfrom __future__ import annotations\n\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef make_review_pack(rows, A, risk, out_dir, attn_dir, n_each=30, seed=0):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from PIL import Image\n\n    out_dir = Path(out_dir)\n    (out_dir / "overlays").mkdir(parents=True, exist_ok=True)\n    idx = [i for i, r in enumerate(rows) if r["split"] == "test" and r["kind"] == "abnormal"\n           and not r["negated"] and not r["uncertain"] and r["label"] is not None]\n    idx.sort(key=lambda i: -risk[i])\n    rng = np.random.default_rng(seed)\n    hi = idx[: n_each]\n    lo = idx[-n_each:]\n    pick = list(hi) + [i for i in lo if i not in hi]\n    rng.shuffle(pick)  # reviewers should not see the ranking order\n    G = int(round(np.sqrt(A["Pl"].shape[-1])))\n    with open(out_dir / "review_sheet.csv", "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["case", "image_id", "generated_sentence", "finding", "location",\n                    "Q1_finding_present_on_image(yes/no/unsure)", "Q2_location_correct(yes/no/na)",\n                    "Q3_highlighted_region_helps_decide(yes/no)", "seconds_to_decide", "notes"])\n        for case, i in enumerate(pick):\n            r = rows[i]\n            meta = json.loads((Path(attn_dir) / f"{r[\'image_id\']}.json").read_text())\n            img = Image.open(meta["image_path"]).convert("L").resize((512, 512))\n            att = A["Pl"][i].astype(np.float32).mean(0).reshape(G, G)\n            fig, ax = plt.subplots(1, 2, figsize=(8, 4))\n            ax[0].imshow(img, cmap="gray")\n            ax[0].set_title("image")\n            ax[1].imshow(img, cmap="gray")\n            ax[1].imshow(np.kron(att, np.ones((512 // G, 512 // G))), cmap="jet", alpha=.4)\n            ax[1].contour(np.kron(A["Ms"][i].reshape(G, G), np.ones((512 // G, 512 // G))), [0.5], colors="w")\n            ax[1].set_title("model attention + named region")\n            for a in ax:\n                a.axis("off")\n            fig.suptitle(r["sentence"][:110], fontsize=8)\n            fig.savefig(out_dir / "overlays" / f"case_{case:03d}.png", dpi=110)\n            plt.close(fig)\n            w.writerow([case, r["image_id"], r["sentence"], r["finding"], r["location_text"], "", "", "", "", ""])\n    # answer key kept separate so reviewers stay blind\n    with open(out_dir / "review_key.csv", "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["case", "label_from_reference", "label_reason", "vera_risk"])\n        for case, i in enumerate(pick):\n            w.writerow([case, rows[i]["label"], rows[i]["label_reason"], float(risk[i])])\n', 'vera/segment.py': '"""Run chest anatomy segmentation once per image and store zone masks on the grid."""\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .anatomy import seg_zone_masks, segment\n\n\ndef run_segmentation(items, out_dir, G=32, device="cuda", log=print):\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    n_ok = n_bad = 0\n    t0 = time.time()\n    for k, it in enumerate(items):\n        f = out_dir / f"{it[\'image_id\']}.npz"\n        if f.exists():\n            continue\n        try:\n            zones = seg_zone_masks(segment(it["image_path"], device=device), G)\n        except Exception as e:  # noqa\n            log(f"seg failed {it[\'image_id\']}: {e!r}"[:200])\n            zones = None\n        if zones is None:\n            np.savez_compressed(f, valid=np.array(False))\n            n_bad += 1\n        else:\n            np.savez_compressed(f, valid=np.array(True), **zones)\n            n_ok += 1\n        if (k + 1) % 100 == 0:\n            log(f"[seg {k + 1}/{len(items)}] {(time.time() - t0) / 60:.1f} min")\n    log(f"segmentation: {n_ok} valid, {n_bad} rejected (not frontal-looking or failed)")\n', 'vera/setup_env.py': '"""Install the pinned stack CheXagent-2-3b needs (transformers==4.40.0).\n\nStrategy that worked on Colab (Python 3.13), kept as the fallback:\n  1. try the clean pinned install (tokenizers 0.19.1 wheels exist for <=3.12)\n  2. otherwise: install a modern tokenizers, then transformers==4.40.0 --no-deps,\n     and neutralise transformers\' strict dependency version check.\n"""\nfrom __future__ import annotations\n\nimport importlib\nimport shutil\nimport subprocess\nimport sys\n\nBASE = ["albumentations", "opencv-python-headless", "einops", "pyarrow", "regex", "tiktoken",\n        "matplotlib", "pandas", "kagglehub", "torchxrayvision", "scikit-image", "sentencepiece", "scikit-learn"]\n\n\ndef _pip(args):\n    cmds = [[sys.executable, "-m", "pip", "install", "-q", *args]]\n    if shutil.which("uv"):\n        cmds.append(["uv", "pip", "install", "--python", sys.executable, "-q", *args])\n    for c in cmds:\n        r = subprocess.run(c, capture_output=True, text=True)\n        if r.returncode == 0:\n            return True, r.stdout[-500:]\n        last = r.stderr[-1500:]\n    return False, last\n\n\ndef _patch_dep_check():\n    import importlib.util, os\n    spec = importlib.util.find_spec("transformers")\n    f = os.path.join(os.path.dirname(spec.origin), "dependency_versions_check.py")\n    with open(f, "w") as fh:\n        fh.write("def dep_version_check(pkg, hint=None):\\n    pass\\n")\n\n\ndef ensure(log=print):\n    if "transformers" in sys.modules and sys.modules["transformers"].__version__ != "4.40.0":\n        log("transformers is already imported at another version -> restart the kernel, then rerun setup.")\n    ok, msg = _pip(BASE)\n    log("base packages:", "ok" if ok else msg)\n    ok, msg = _pip(["transformers==4.40.0", "tokenizers==0.19.1", "huggingface_hub==0.25.2"])\n    if not ok:\n        log("pinned install failed, using the --no-deps fallback")\n        _pip(["tokenizers>=0.19", "huggingface_hub==0.25.2", "safetensors"])\n        ok, msg = _pip(["transformers==4.40.0", "--no-deps", "--force-reinstall"])\n        if not ok:\n            raise RuntimeError(msg)\n        _patch_dep_check()\n    importlib.invalidate_caches()\n    import transformers\n    log("transformers", transformers.__version__)\n    import torch\n    log("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),\n        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")\n    return transformers.__version__ == "4.40.0"\n', 'vera/text.py': '"""Sentence splitting, claim extraction and negation for radiology text.\n\nFixes over v1:\n  * CheXagent reports look like "[Breathing: Lungs] **Text.** [Cardiac: ...] Text."\n    v1\'s splitter needed an uppercase letter after the period, so whole reports\n    came back as ONE sentence and every finding was paired with every location.\n    Here tags and markdown are treated as boundaries and every sentence keeps its\n    character span in the ORIGINAL text (needed for exact token alignment).\n  * Locations are paired with the nearest finding inside the same clause,\n    not as a cross-product.\n  * Negation is scoped (trigger shortly before the finding, clause-bounded, plus\n    post-negations like "is not seen"). v1 negated anything after any "no".\n  * Hedged statements ("possible", "cannot exclude") are flagged as uncertain.\n"""\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass, field, asdict\nfrom typing import List, Optional, Tuple\n\n# --------------------------------------------------------------------------\n# Sentence splitting with offsets into the original string\n# --------------------------------------------------------------------------\n# [Breathing: Lungs] section tags, CheXagent grounding tokens <|ref|> <|/ref|>,\n# and <|box|> (x,y),(x,y) <|/box|> coordinate blocks are all sentence boundaries.\n_TAG = re.compile(r"\\[[^\\]]*\\]|<\\|box\\|>.*?<\\|/box\\|>|<\\|[^|]{1,12}\\|>")\n_SENT_END = re.compile(r"(?<=[.!?])\\s+")\n\n\n@dataclass\nclass Sentence:\n    text: str\n    start: int   # char offset in original text\n    end: int\n\n\ndef split_sentences(text: str) -> List[Sentence]:\n    """Split into sentences, dropping [tags] and ** markers, keeping offsets."""\n    out: List[Sentence] = []\n    if not text:\n        return out\n    # chunks between tags\n    cuts, last = [], 0\n    for m in _TAG.finditer(text):\n        cuts.append((last, m.start()))\n        last = m.end()\n    cuts.append((last, len(text)))\n    for a, b in cuts:\n        chunk = text[a:b]\n        pos = 0\n        for piece in _SENT_END.split(chunk):\n            idx = chunk.find(piece, pos)\n            if idx < 0:\n                continue\n            pos = idx + len(piece)\n            s, e = a + idx, a + idx + len(piece)\n            # trim whitespace and markdown asterisks while tracking offsets\n            while s < e and (text[s].isspace() or text[s] == "*"):\n                s += 1\n            while e > s and (text[e - 1].isspace() or text[e - 1] == "*"):\n                e -= 1\n            if e - s >= 4 and re.search(r"[A-Za-z]", text[s:e]):\n                out.append(Sentence(text[s:e], s, e))\n    return out\n\n\n# --------------------------------------------------------------------------\n# Findings lexicon: canonical name -> (regex, severity tier, default location)\n# --------------------------------------------------------------------------\nFINDINGS = {\n    # critical\n    "pneumothorax":       (r"pneumothora(?:x|ces)", "critical", "lungs"),\n    "pleural_effusion":   (r"(?:pleural\\s+)?effusions?|blunting\\s+of\\s+the\\s+(?:\\w+\\s+){0,2}costophrenic|costophrenic\\s+(?:angle\\s+)?blunting",\n                           "critical", "costophrenic"),\n    "mass":               (r"\\bmass(?:es)?\\b", "critical", None),\n    "pneumonia":          (r"pneumonias?", "critical", "lungs"),\n    "edema":              (r"(?:pulmonary\\s+|interstitial\\s+|alveolar\\s+)?edema", "critical", "lungs"),\n    "fracture":           (r"fractures?", "critical", None),\n    # moderate\n    "consolidation":      (r"consolidat(?:ion|ions|ive)", "moderate", "lungs"),\n    "opacity":            (r"opacit(?:y|ies)|opacification|airspace\\s+(?:disease|opacit\\w*)", "moderate", "lungs"),\n    "infiltrate":         (r"infiltrat(?:e|es|ion|ive)", "moderate", "lungs"),\n    "atelectasis":        (r"atelecta(?:sis|tic)", "moderate", "lungs"),\n    "cardiomegaly":       (r"cardiomegaly|(?:heart|cardiac\\s+silhouette|cardiac\\s+size|heart\\s+size)\\s+(?:is\\s+|appears\\s+|remains\\s+)?(?:\\w+\\s+)?(?:enlarged|enlargement)|enlarged\\s+(?:heart|cardiac\\s+silhouette)",\n                           "moderate", "heart"),\n    "mediastinal_widening": (r"mediastinal\\s+widening|widen(?:ed|ing)\\s+(?:of\\s+the\\s+)?mediastinum", "moderate", "mediastinum"),\n    "nodule":             (r"nodul(?:e|es|ar)", "moderate", None),\n    "vascular_congestion": (r"(?:pulmonary\\s+)?vascular\\s+congestion|cephalization", "moderate", "lungs"),\n    "pleural_thickening": (r"pleural\\s+thickening", "moderate", "pleural"),\n    # mild\n    "granuloma":          (r"granulomas?|granulomatous", "mild", None),\n    "calcification":      (r"calcifications?|calcified\\s+(?:lymph\\s+)?nodes?", "mild", None),\n    "scarring":           (r"scarring|\\bscar\\b|fibrosis|fibrotic", "mild", "lungs"),\n    "hyperinflation":     (r"hyperinflat\\w+|hyperexpan\\w+|emphysema\\w*|copd", "mild", "lungs"),\n    "tortuous_aorta":     (r"tortuous\\s+(?:thoracic\\s+)?aorta|aortic\\s+tortuosity|ectatic\\s+aorta|unfolded\\s+aorta|aorta\\s+is\\s+(?:\\w+\\s+){0,3}tortuous|aortic\\s+enlargement|(?:enlarged|dilated)\\s+aorta", "mild", "mediastinum"),\n    "degenerative":       (r"degenerative\\s+(?:changes?|disease)|spondylosis|osteophyt\\w+", "mild", None),\n    "scoliosis":          (r"scoliosis|curvature\\s+of\\s+the\\s+spine", "mild", None),\n}\n_FINDING_RE = {k: re.compile(r"\\b(?:" + v[0] + r")", re.I) for k, v in FINDINGS.items()}\nTIER = {k: v[1] for k, v in FINDINGS.items()}\nDEFAULT_LOC = {k: v[2] for k, v in FINDINGS.items()}\n\n# Normal statements -> organ\nNORMAL_PATTERNS = [\n    (re.compile(r"\\blungs?\\s+(?:are|is|appear|remain)\\s+(?:\\w+\\s+)?clear|\\bclear\\s+lungs?\\b|lungs?\\s+(?:are\\s+)?(?:well\\s+)?expanded\\s+and\\s+clear", re.I), "lungs"),\n    (re.compile(r"\\b(?:heart|cardiac)\\s+(?:size|silhouette)?\\s*(?:is|are|appears?|remains?)\\s+(?:\\w+\\s+)?(?:normal|within\\s+normal|unremarkable)", re.I), "heart"),\n    (re.compile(r"cardiomediastinal\\s+(?:silhouette|contours?)\\s+(?:is|are|appears?)\\s+(?:\\w+\\s+)?(?:normal|within\\s+normal|unremarkable|stable)", re.I), "cardiomediastinal"),\n    (re.compile(r"mediastinal\\s+(?:contours?|silhouette)\\s+(?:is|are|appears?)\\s+(?:\\w+\\s+)?(?:normal|within\\s+normal|unremarkable)", re.I), "mediastinum"),\n]\n\n# --------------------------------------------------------------------------\n# Location parsing: laterality x region -> zone names\n# --------------------------------------------------------------------------\n_LOC_RE = re.compile(\n    r"\\b(?:(right|left|bilateral(?:ly)?|bibasilar|bibasal|both)\\s+)?"\n    r"(?:(upper|middle|mid|lower)\\s+)?"\n    r"(lobes?|lungs?|lung\\s+fields?|lung\\s+bases?|bases?|basilar|basal|apex|apices|apical|"\n    r"midlung|mid\\s+lung|hil(?:um|a|ar)|perihilar|costophrenic(?:\\s+angles?)?|"\n    r"hemidiaphragms?|diaphragms?|pleural|lingula(?:r)?|retrocardiac|"\n    r"heart|cardiac(?:\\s+silhouette)?|cardiomediastinal|mediastin(?:um|al)|aort(?:a|ic)|trachea)\\b",\n    re.I,\n)\n# also bare laterality words like "right-sided effusion" / "on the left"\n_SIDE_RE = re.compile(r"\\b(right|left|bilateral(?:ly)?|bibasilar)\\b", re.I)\n\nZONES_ALL_LUNG = ["right_upper_lobe", "right_middle_lobe", "right_lower_lobe",\n                  "left_upper_lobe", "left_middle_zone", "left_lower_lobe"]\n\n\ndef _sides(lat: Optional[str]) -> List[str]:\n    if not lat:\n        return ["right", "left"]\n    lat = lat.lower()\n    if lat.startswith("right"):\n        return ["right"]\n    if lat.startswith("left"):\n        return ["left"]\n    return ["right", "left"]\n\n\ndef location_to_zones(lat: Optional[str], level: Optional[str], region: str) -> List[str]:\n    region = region.lower()\n    sides = _sides(lat)\n    lv = (level or "").lower()\n    z: List[str] = []\n    if region.startswith(("lobe", "lung", "midlung", "mid lung")):\n        for s in sides:\n            if region.startswith(("midlung", "mid lung")) or lv in ("middle", "mid"):\n                z.append(f"{s}_middle_lobe" if s == "right" else "left_middle_zone")\n            elif lv == "upper":\n                z.append(f"{s}_upper_lobe")\n            elif lv == "lower":\n                z.append(f"{s}_lower_lobe")\n            else:\n                z += [f"{s}_upper_lobe", f"{s}_middle_lobe" if s == "right" else "left_middle_zone", f"{s}_lower_lobe"]\n        if "base" in region:\n            z = [f"{s}_lower_lobe" for s in sides]\n    elif region.startswith(("base", "basilar", "basal")) or lat and lat.lower().startswith(("bibas",)):\n        z = [f"{s}_lower_lobe" for s in sides]\n    elif region.startswith(("apex", "apices", "apical")):\n        z = [f"{s}_apex" for s in sides]\n    elif region.startswith(("hil", "perihilar")):\n        z = [f"{s}_hilum" for s in sides]\n    elif region.startswith("costophrenic"):\n        z = [f"{s}_costophrenic_angle" for s in sides]\n    elif region.startswith(("hemidiaphragm", "diaphragm")):\n        z = [f"{s}_hemidiaphragm" for s in sides]\n    elif region.startswith("pleural"):\n        z = [f"{s}_costophrenic_angle" for s in sides] + [f"{s}_lower_lobe" for s in sides]\n    elif region.startswith("lingula"):\n        z = ["left_middle_zone"]\n    elif region.startswith("retrocardiac"):\n        z = ["left_lower_lobe", "cardiac_silhouette"]\n    elif region.startswith(("heart", "cardiac")) and not region.startswith("cardiomediastinal"):\n        z = ["cardiac_silhouette"]\n    elif region.startswith("cardiomediastinal"):\n        z = ["cardiac_silhouette", "mediastinum"]\n    elif region.startswith(("mediastin", "aort", "trachea")):\n        z = ["mediastinum"]\n    return list(dict.fromkeys(z))\n\n\ndef default_zones(name: Optional[str]) -> List[str]:\n    if name is None:\n        return []\n    return {\n        "lungs": ZONES_ALL_LUNG,\n        "costophrenic": ["right_costophrenic_angle", "left_costophrenic_angle"],\n        "pleural": ["right_costophrenic_angle", "left_costophrenic_angle", "right_lower_lobe", "left_lower_lobe"],\n        "heart": ["cardiac_silhouette"],\n        "mediastinum": ["mediastinum"],\n        "cardiomediastinal": ["cardiac_silhouette", "mediastinum"],\n    }[name]\n\n\n# --------------------------------------------------------------------------\n# Negation / uncertainty (NegEx-lite, clause scoped)\n# --------------------------------------------------------------------------\n_PRE_NEG = re.compile(r"\\b(?:no|not|without|negative\\s+for|free\\s+of|clear\\s+of|absence\\s+of|"\n                      r"resolution\\s+of|resolved|removed|rules?\\s+out|ruled\\s+out|nor)\\b", re.I)\n_POST_NEG = re.compile(r"^\\W*(?:\\w+\\W+){0,3}?(?:is|are|was|were)\\s+(?:not\\s+(?:seen|identified|present|visualized|evident|demonstrated|appreciated)|absent|resolved)", re.I)\n_UNCERTAIN = re.compile(r"\\b(?:possib(?:le|ly)|probab(?:le|ly)|likely|may|might|could|questionable|represent\\w*|indicat\\w*|"\n                        r"cannot\\s+(?:be\\s+)?(?:excluded|ruled\\s+out)|can\\s?not\\s+exclude|suggest\\w*|"\n                        r"concerning\\s+for|versus|vs\\.?|equivocal|suspicious|borderline)\\b", re.I)\n_RELATIONAL = re.compile(r"\\b(?:interval|compared|comparison|prior|previous(?:ly)?|since|"\n                         r"increas\\w*|decreas\\w*|improv\\w*|worsen\\w*|unchanged|stable|new|resolv\\w*|persistent)\\b", re.I)\n_CLAUSE_BREAK = re.compile(r"[;,:]|\\b(?:but|however|although|though|whereas|except)\\b", re.I)\n\n\ndef _clause_window(text: str, start: int, end: int) -> Tuple[str, str]:\n    """Text before/after the mention, cut at clause boundaries."""\n    before = text[:start]\n    m = list(_CLAUSE_BREAK.finditer(before))\n    if m:\n        before = before[m[-1].end():]\n    after = text[end:]\n    m2 = _CLAUSE_BREAK.search(after)\n    if m2:\n        after = after[:m2.start()]\n    return before, after\n\n\ndef polarity(text: str, start: int, end: int) -> Tuple[bool, bool]:\n    """Return (negated, uncertain) for the mention at text[start:end]."""\n    before, after = _clause_window(text, start, end)\n    words_before = before.split()[-7:]\n    neg = bool(_PRE_NEG.search(" ".join(words_before))) or bool(_POST_NEG.search(after))\n    unc = bool(_UNCERTAIN.search(before)) or bool(_UNCERTAIN.search(after))\n    return neg, unc\n\n\n# --------------------------------------------------------------------------\n# Claim extraction\n# --------------------------------------------------------------------------\n@dataclass\nclass Claim:\n    sentence_idx: int\n    sentence: str\n    sent_start: int\n    sent_end: int\n    kind: str                 # "abnormal" | "normal"\n    finding: str              # canonical finding or "normal"\n    mention: str\n    tier: str                 # critical | moderate | mild\n    negated: bool\n    uncertain: bool\n    relational: bool          # temporal/comparison language without a prior study\n    location_text: str\n    laterality: Optional[str]\n    zones: List[str] = field(default_factory=list)\n\n    @property\n    def localizable(self) -> bool:\n        return len(self.zones) > 0\n\n    def to_dict(self):\n        d = asdict(self)\n        d["localizable"] = self.localizable\n        return d\n\n\ndef _locations(sentence: str):\n    out = []\n    for m in _LOC_RE.finditer(sentence):\n        lat, lvl, reg = m.group(1), m.group(2), m.group(3)\n        zones = location_to_zones(lat, lvl, reg)\n        if zones:\n            out.append((m.start(), m.end(), m.group(0), lat, zones))\n    return out\n\n\ndef extract_claims(report: str) -> List[Claim]:\n    claims: List[Claim] = []\n    for si, s in enumerate(split_sentences(report)):\n        t = s.text\n        locs = _locations(t)\n        found_any = False\n        taken: List[Tuple[int, int]] = []\n        for name, rx in _FINDING_RE.items():\n            for m in rx.finditer(t):\n                if any(not (m.end() <= a or m.start() >= b) for a, b in taken):\n                    continue  # overlapping a longer/earlier match\n                taken.append((m.start(), m.end()))\n                found_any = True\n                neg, unc = polarity(t, m.start(), m.end())\n                # nearest location within the same clause\n                zones, loc_text, lat = [], "", None\n                best = None\n                for (a, b, txt, la, zz) in locs:\n                    lo, hi = (min(b, m.start()), max(a, m.end()))\n                    between = t[lo:hi]\n                    if _CLAUSE_BREAK.search(between) and not (a <= m.start() <= b):\n                        continue\n                    d = 0 if (a <= m.start() < b or m.start() <= a < m.end()) else min(abs(a - m.end()), abs(m.start() - b))\n                    if best is None or d < best[0]:\n                        best = (d, txt, la, zz)\n                if best is not None:\n                    _, loc_text, lat, zones = best\n                else:\n                    side = _SIDE_RE.search(t)\n                    dz = default_zones(DEFAULT_LOC[name])\n                    if side and side.group(1).lower().startswith("bibas"):\n                        dz, lat = ["right_lower_lobe", "left_lower_lobe"], "bibasilar"\n                    elif side and dz:\n                        keep = _sides(side.group(1))\n                        dz = [z for z in dz if z.split("_")[0] in keep] or dz\n                        lat = side.group(1)\n                    zones, loc_text = dz, (DEFAULT_LOC[name] or "")\n                rel = bool(_RELATIONAL.search(t))\n                claims.append(Claim(si, t, s.start, s.end, "abnormal", name, m.group(0), TIER[name],\n                                    neg, unc, rel, loc_text, lat, zones))\n        if not found_any:\n            for rx, organ in NORMAL_PATTERNS:\n                m = rx.search(t)\n                if m:\n                    claims.append(Claim(si, t, s.start, s.end, "normal", "normal", m.group(0), "mild",\n                                        False, False, False, organ, None, default_zones(organ)))\n                    break\n    return claims\n'}
for _rel, _src in PKG.items():
    (Path("vera_src") / _rel).parent.mkdir(parents=True, exist_ok=True)
    (Path("vera_src") / _rel).write_text(_src)
sys.path.insert(0, str(Path("vera_src").resolve()))
pkg_ok = True
print("vera package written:", len(PKG), "modules")

In [ ]:
from vera import setup_env
env_ok = setup_env.ensure()
assert env_ok, "transformers 4.40.0 not active: restart the kernel and run all again"


In [ ]:
# settings
N_TOTAL = 3000        # frontal studies (IU has ~3.8k; 75% abnormal reference, rest normal)
MAX_TOKENS = 256
from vera.pipeline import Paths
P = Paths("vera_work")
print("work dir:", P.work)

In [ ]:
from vera.pipeline import stage_data
items = stage_data(P, n_total=N_TOTAL)

In [ ]:
# smoke test: 3 images. Check grid 32x32 and the heat map sits on the chest.
import json as _json
import numpy as _np
import matplotlib.pyplot as _plt
from PIL import Image as _Image
from vera.pipeline import stage_extract
stage_extract(P, max_new_tokens=MAX_TOKENS, limit=3)
_metas = [P.attn / f"{it['image_id']}.json" for it in items[:3]]
smoke_fig, _ax = _plt.subplots(1, 3, figsize=(15, 5))
for _a, _m in zip(_ax, _metas):
    _meta = _json.loads(_m.read_text())
    _lm = _np.load(_m.with_suffix(".npz"))["layer_map"].astype(float)
    _G = _meta["grid"]
    _att = _lm[:, _lm.shape[1] // 2].mean(0)
    _a.imshow(_Image.open(_meta["image_path"]).convert("L").resize((512, 512)), cmap="gray")
    _a.imshow(_np.kron(_att / _att.max(), _np.ones((512 // _G, 512 // _G))), cmap="jet", alpha=0.35)
    _a.set_title(f"grid {_G}x{_G}, {_meta['seconds']} s", fontsize=9)
    _a.axis("off")
    print(_meta["report"][:300], "\n")
_plt.close(smoke_fig)
smoke_fig

In [ ]:
# full extraction (resumable; prints progress + ETA every 10 images)
fails = stage_extract(P, max_new_tokens=MAX_TOKENS)
print("failures:", len(fails))

In [ ]:
import torch
from vera.pipeline import stage_segment
stage_segment(P, device="cuda" if torch.cuda.is_available() else "cpu")
seg_done = True

In [ ]:
from vera.pipeline import stage_analyze
from vera.analysis import summary_md
R = stage_analyze(P)
print(summary_md(R))

In [ ]:
from vera.pipeline import stage_bundle
bundle = stage_bundle(P)
try:
    from google.colab import files as _files
    _files.download(str(bundle))
except Exception:
    print("download this file and send it to Claude:", bundle)